In [1]:
import uuid
from typing import TypedDict, Optional

from langgraph.graph import StateGraph
from langgraph.constants import START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


In [2]:
class State(TypedDict):
    username: Optional[str]
    email: Optional[str]


In [3]:
def collect_user_info(state: State) -> State:
    # Interrupt 1: Ask for username if not provided
    if not state.get("username"):
        username = interrupt("Please enter your username:")
    else:
        username = state["username"]

    # Interrupt 2: Ask for email if not provided
    if not state.get("email"):
        email = interrupt("Please enter your email:")
    else:
        email = state["email"]

    print(f"Collected Username: {username}")
    print(f"Collected Email: {email}")

    return {
        "username": username,
        "email": email,
    }


In [4]:
builder = StateGraph(State)
builder.add_node("collect_user_info", collect_user_info)
builder.add_edge(START, "collect_user_info")
builder.add_edge("collect_user_info", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)


In [5]:
config = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

# First call: both fields are missing
for chunk in graph.stream({"username": None, "email": None}, config):
    print(chunk)

# First resume: supply username
for chunk in graph.stream(Command(resume="alice"), config):
    print(chunk)

# Second resume: supply email
for chunk in graph.stream(Command(resume="alice@example.com"), config):
    print(chunk)


{'__interrupt__': (Interrupt(value='Please enter your username:', resumable=True, ns=['collect_user_info:5119858e-7b54-c0da-8231-8a8c442ff865']),)}
{'__interrupt__': (Interrupt(value='Please enter your email:', resumable=True, ns=['collect_user_info:5119858e-7b54-c0da-8231-8a8c442ff865']),)}
Collected Username: alice
Collected Email: alice@example.com
{'collect_user_info': {'username': 'alice', 'email': 'alice@example.com'}}


In [11]:
# First call: both fields are missing
for chunk in graph.stream({"username": None, "email": None}, config):
    print(chunk)

{'__interrupt__': (Interrupt(value='Please enter your username:', resumable=True, ns=['collect_user_info:c06eb990-d191-6ea2-0aa8-8be60d9d66a6']),)}


In [12]:
for chunk in graph.stream(Command(resume="alice", update={"username": "already_set"}), config):
    print(chunk)


Collected Username: already_set
Collected Email: alice
{'collect_user_info': {'username': 'already_set', 'email': 'alice'}}


In [13]:
# First call: both fields are missing
for chunk in graph.stream({"username": "pankaj", "email": "pankaj@gmail.com"}, config):
    print(chunk)

Collected Username: pankaj
Collected Email: pankaj@gmail.com
{'collect_user_info': {'username': 'pankaj', 'email': 'pankaj@gmail.com'}}


In [14]:
# First call: both fields are missing
for chunk in graph.stream({"username": "pankaj", "email": None}, config):
    print(chunk)

{'__interrupt__': (Interrupt(value='Please enter your email:', resumable=True, ns=['collect_user_info:dc76105a-002a-fd7d-72c9-0821859251ff']),)}
